In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
df = pd.read_csv('spam.csv',encoding='latin1')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


In [4]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
x = df['v2']
y = df['v1']

xtrain,xtest,ytrain,ytest = train_test_split(x,y,random_state=42,train_size=0.8)

In [6]:
main_pipeline = Pipeline(
    steps=[
        ('vectorizer',TfidfVectorizer()),
        ('model',LogisticRegression())
    ]
)

main_pipeline.fit(xtrain,ytrain)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vectorizer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['ham','spam']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


## Model evaluation

In [7]:
## Train Data
train_score = classification_report(ytrain,main_pipeline.predict(xtrain))

## Test data
test_score = classification_report(ytest,main_pipeline.predict(xtest))

print(f'----------------Train Data Performance----------------')
print(train_score)
print(f'----------------Test Data Performance----------------')
print(test_score)

----------------Train Data Performance----------------
              precision    recall  f1-score   support

         ham       0.97      1.00      0.99      3860
        spam       1.00      0.82      0.90       597

    accuracy                           0.98      4457
   macro avg       0.98      0.91      0.94      4457
weighted avg       0.98      0.98      0.98      4457

----------------Test Data Performance----------------
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       965
        spam       0.99      0.77      0.86       150

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



# So the model is good fit and generalized well(captures unseen data patterns)

## future prediction

In [8]:
df['v2'][0]

'Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...'

In [9]:
df['v2'][1]

'Ok lar... Joking wif u oni...'

In [10]:
new_data = pd.DataFrame({
    'v2':['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...','Ok lar... Joking wif u oni...']
})

new_data

,v2
0,"Go until jurong point, crazy.. Available only ..."
1,Ok lar... Joking wif u oni...


In [11]:
result = main_pipeline.predict(new_data['v2'])

In [12]:
import os
llm_model = ChatGoogleGenerativeAI(
    model = 'gemini-3.5-flash',
    api_key = os.environ[''GOOGLE_API_KEY''],
)

In [13]:
prompt = f'''
You're an AI assistant.
Scenario:
Write the conclusion for my given input and output.
input = {new_data}
output = {result}
Note:
Use serial numbers for each input like 1),2),...
Explanation must be simple.
Output Format:
Input:
Output:
Explanation:
'''
result = llm_model.invoke(prompt)

In [14]:
print(result.content[0]['text'])

**Input:**
1) Go until jurong point, crazy.. Available only ...
2) Ok lar... Joking wif u oni...

**Output:**
1) ham
2) ham

**Explanation:**
In message classification, "ham" means a normal or legitimate message (not spam). Both inputs are regular conversational text messages, so the system correctly classified both of them as "ham".
